In [18]:
# Install ultralytics package if not already installed
!pip install ultralytics


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from dataclasses import dataclass

# =========================================================
# Configuration
# =========================================================
@dataclass
class Config:
    input_video: str = "input.mp4"
    output_video: str = "output_suspicious.mp4"

    # Use YOLOv8 pose model (keypoints included)
    pose_model: str = "yolov8n-pose.pt"

    # Detection thresholds
    conf: float = 0.35
    iou: float = 0.45

    # Performance / safety
    max_people: int = 10

    # Visualization
    show: bool = False
    draw_keypoints: bool = True


# =========================================================
# YOLOv8-Pose keypoint indices (COCO-17)
# =========================================================
# 0:nose
# 5:left_shoulder, 6:right_shoulder
# 9:left_wrist, 10:right_wrist
# 11:left_hip, 12:right_hip
# 13:left_knee, 14:right_knee
KP = {
    "NOSE": 0,
    "L_SHOULDER": 5,
    "R_SHOULDER": 6,
    "L_WRIST": 9,
    "R_WRIST": 10,
    "L_HIP": 11,
    "R_HIP": 12,
    "L_KNEE": 13,
    "R_KNEE": 14,
}


# =========================================================
# Suspicious activity logic (same rules, more readable)
# =========================================================
def avg_y(kpts_xy, a, b):
    """Average y-coordinate of two keypoints."""
    return (kpts_xy[a][1] + kpts_xy[b][1]) / 2.0

def is_hands_up(kpts_xy):
    """
    Hands up:
    If both wrists are above the nose.
    (In image coords, smaller y = higher on screen)
    """
    nose_y = kpts_xy[KP["NOSE"]][1]
    lw_y = kpts_xy[KP["L_WRIST"]][1]
    rw_y = kpts_xy[KP["R_WRIST"]][1]
    return (lw_y < nose_y) and (rw_y < nose_y)

def is_crouching(hip_y, shoulder_y, knee_y):
    """
    Crouching:
    torso_len = |hip - shoulder|
    thigh_len = |knee - hip|
    suspicious if thigh_len < 0.4 * torso_len
    """
    torso_len = abs(hip_y - shoulder_y) + 1e-6
    thigh_len = abs(knee_y - hip_y)
    return thigh_len < 0.4 * torso_len

def is_lying_down(hip_y, shoulder_y, knee_y, eps):
    """
    Lying down:
    If shoulder, hip, knee are nearly at same vertical level.
    (Difference < eps)
    """
    return (
        abs(hip_y - shoulder_y) < eps and
        abs(knee_y - hip_y) < eps and
        abs(knee_y - shoulder_y) < eps
    )

def is_crawling(hip_y, shoulder_y, knee_y, eps):
    """
    Crawling:
    - knees near hips vertically (|hip - knee| < eps)
    - shoulders below hips (shoulder_y > hip_y)
    """
    return (abs(hip_y - knee_y) < eps) and (shoulder_y > hip_y)

def is_fall(nose_y, hip_y):
    """
    Fall:
    If nose is below hips (nose_y > hip_y).
    """
    return nose_y > hip_y

def detect_suspicious_activity(kpts_xy, frame_height):
    """
    Apply the rules in the SAME order as your original function.
    Returns (suspicious: bool, label: str)
    """
    # Convert your normalized threshold (~0.05) into pixel threshold:
    eps = 0.05 * frame_height

    # Key y-levels
    nose_y = kpts_xy[KP["NOSE"]][1]
    hip_y = avg_y(kpts_xy, KP["L_HIP"], KP["R_HIP"])
    shoulder_y = avg_y(kpts_xy, KP["L_SHOULDER"], KP["R_SHOULDER"])
    knee_y = avg_y(kpts_xy, KP["L_KNEE"], KP["R_KNEE"])

    # Rule 1: Hands Up
    if is_hands_up(kpts_xy):
        return True, "Hands Up Detected"

    # Rule 2: Crouching
    if is_crouching(hip_y, shoulder_y, knee_y):
        return True, "Crouching Detected"

    # Rule 3: Lying Down
    if is_lying_down(hip_y, shoulder_y, knee_y, eps):
        return True, "Lying Down Detected"

    # Rule 4: Crawling
    if is_crawling(hip_y, shoulder_y, knee_y, eps):
        return True, "Crawling Detected"

    # Rule 5: Fall
    if is_fall(nose_y, hip_y):
        return True, "Fall Detected"

    return False, ""


# =========================================================
# Video processing loop
# =========================================================
def run_suspicious_detection(cfg: Config):
    model = YOLO(cfg.pose_model)

    cap = cv2.VideoCapture(cfg.input_video)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open input video: {cfg.input_video}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 1e-6:
        fps = 25.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        cfg.output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (W, H)
    )

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # Pose detection (people + keypoints)
        res = model.predict(frame, conf=cfg.conf, iou=cfg.iou, verbose=False)[0]

        if res.boxes is not None and len(res.boxes) > 0 and res.keypoints is not None:
            # Select top-N people by confidence
            confs = res.boxes.conf.cpu().numpy()
            top_idx = np.argsort(-confs)[:cfg.max_people]

            boxes = res.boxes.xyxy.cpu().numpy().astype(int)     # (N,4)
            kpts = res.keypoints.xy.cpu().numpy()                # (N,17,2) pixels

            for i in top_idx:
                x1, y1, x2, y2 = boxes[i].tolist()
                kpts_i = kpts[i]  # (17,2)

                suspicious, label = detect_suspicious_activity(kpts_i, H)

                # Colors: red for suspicious, green otherwise
                color = (0, 0, 255) if suspicious else (0, 255, 0)
                text = label if suspicious else "Person"

                # Draw bounding box + label
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame, text,
                    (x1, max(20, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, color, 2, cv2.LINE_AA
                )

                # Optional: draw keypoints
                if cfg.draw_keypoints:
                    for (x, y) in kpts_i.astype(int):
                        cv2.circle(frame, (x, y), 2, color, -1)

        writer.write(frame)

        if cfg.show:
            cv2.imshow("Suspicious Activity Detection (YOLO Pose)", frame)
            if cv2.waitKey(1) & 0xFF == 27:
                break

    cap.release()
    writer.release()
    cv2.destroyAllWindows()
    print("Saved output video to:", cfg.output_video)


# =========================================================
# Run for your input.mp4
# =========================================================
cfg = Config(
    input_video="input.mp4",
    output_video="output_suspicious.mp4",
    pose_model="yolov8n-pose.pt",
    conf=0.35,
    iou=0.45,
    show=False,
    draw_keypoints=True
)

run_suspicious_detection(cfg)

RuntimeError: Failed to open input video: input.mp4